### Балансовый долг

- **STD** — краткосрочные заимствования (например, коммерческие бумаги, кредиты и РЕПО) плюс текущая часть долгосрочного долга. Если отчёт уже содержит общий `DebtCurrent`, берём эту сумму целиком.
- **LTD** — нетекущая часть долгосрочного долга. Текущую и нетекущую части берём по классификации эмитента в балансе, обычно с границей в 12 месяцев. Обе колонки содержат балансовые остатки в USD.
- Финансовая аренда входит в долг в той мере, в какой эмитент включил её в строку долга (как Boeing). Отдельно раскрытая аренда, кредиторская задолженность, налоговые и пенсионные обязательства остаются за пределами этих колонок.

`Date` — дата подачи исходного 10-Q или 10-K. Компоненты долга берутся на дату баланса из этой же подачи. Для `Shares` берём число с титульной страницы, при его отсутствии — общее число акций на дату баланса.

При отсутствии общего тега `STD` складывается из доступных компонентов. Для текущей части долгосрочного долга запасной тег — `ConvertibleDebtCurrent`. Company Facts содержит стандартные теги всего эмитента, поэтому сумма по компонентам может быть неполной, а отсутствие данных в CSV обозначается пустой ячейкой.

Исторические строки Apple сверены по [10-Q за июнь 2013](https://www.sec.gov/Archives/edgar/data/320193/000119312513300670/d552802d10q.htm), [10-K за 2014](https://www.sec.gov/Archives/edgar/data/320193/000119312514383437/d783162d10k.htm) и [балансам за март](https://www.sec.gov/Archives/edgar/data/320193/000119312515153166/R4.htm) и [июнь 2015](https://www.sec.gov/Archives/edgar/data/320193/000119312515259935/R4.htm).

In [12]:
import pandas as pd
import requests
import re
import time

In [13]:
def download_credit_data(companies, email, *, save_csv=True):
    s = requests.Session()
    s.headers.update({"User-Agent": f"Research {email}"})

    tickers = s.get("https://www.sec.gov/files/company_tickers.json", timeout=30)
    tickers.raise_for_status()
    lookup = {
        v["ticker"].upper(): str(v["cik_str"]).zfill(10)
        for v in tickers.json().values()
    }

    def extract(facts, tag, group="us-gaap", unit="USD"):
        items = facts.get(group, {}).get(tag, {}).get("units", {}).get(unit, [])
        rows = [
            (x["accn"], x["filed"], x["end"], x["val"])
            for x in items
            if x["form"] in ("10-K", "10-Q") and "start" not in x
        ]
        df = pd.DataFrame(rows, columns=["accn", "filed", "end", "val"])
        df["val"] = pd.to_numeric(df["val"]).convert_dtypes()
        return df

    results = {}
    for company in companies:
        try:
            match = re.search(r"CIK(\d+)", company, re.I)
            cik = match.group(1).zfill(10) if match else lookup[company.upper()]
            name = company.upper() if not match else f"CIK{cik}"

            url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
            response = s.get(url, timeout=30)
            response.raise_for_status()
            facts = response.json()["facts"]

            df = (extract(facts, "Assets").sort_values("end")
                  .drop_duplicates("accn", keep="last").drop(columns="val")
                  .set_index(["accn", "end"]))
            shares = extract(facts, "EntityCommonStockSharesOutstanding", "dei", "shares")
            shares = shares.sort_values("end").drop_duplicates("accn", keep="last")
            df = df.join(shares.set_index("accn")["val"].rename("Shares"), on="accn")

            for tag in (
                "DebtCurrent", "ShortTermBorrowings", "CommercialPaper", "OtherShortTermBorrowings",
                "LongTermDebtAndCapitalLeaseObligationsCurrent", "LongTermDebtCurrent",
                "ConvertibleDebtCurrent", "ConvertibleDebtNoncurrent",
                "LongTermDebtAndCapitalLeaseObligations", "LongTermDebtNoncurrent", "LongTermDebt",
                "CommonStockSharesOutstanding",
            ):
                unit = "shares" if tag == "CommonStockSharesOutstanding" else "USD"
                values = extract(facts, tag, unit=unit)[["accn", "end", "val"]].drop_duplicates()
                values = values.drop_duplicates(["accn", "end"], keep=False)
                df[tag] = values.set_index(["accn", "end"])["val"]
            df["Shares"] = df["Shares"].combine_first(df["CommonStockSharesOutstanding"])

            short = df[["CommercialPaper", "OtherShortTermBorrowings"]].sum(axis=1, min_count=1)
            short = df["ShortTermBorrowings"].combine_first(short)
            current = df["LongTermDebtAndCapitalLeaseObligationsCurrent"].combine_first(
                df["LongTermDebtCurrent"]
            ).combine_first(df["ConvertibleDebtCurrent"])
            std = pd.concat([short, current], axis=1).sum(axis=1, min_count=1)
            df["STD"] = df["DebtCurrent"].combine_first(std)
            df["LTD"] = df["LongTermDebtAndCapitalLeaseObligations"].combine_first(
                df["LongTermDebtNoncurrent"]
            )

            if cik == "0000320193":

                end = df.index.get_level_values("end")
                df["LTD"] = df["LTD"].combine_first(df["LongTermDebt"].where(end < "2015-06-27"))
                df.loc[end < "2013-06-29", "LTD"] = 0
                df.loc[end < "2014-06-28", "STD"] = 0

            incomplete = df["LTD"].lt(df["ConvertibleDebtNoncurrent"]).fillna(False)
            df.loc[incomplete, "LTD"] = pd.NA

            df = (df.reset_index().sort_values(["filed", "end"])
                  .drop_duplicates("filed", keep="last")
                  .rename(columns={"filed": "Date"})[["Date", "Shares", "STD", "LTD"]]
                  .reset_index(drop=True))
            df["Date"] = pd.to_datetime(df["Date"])

            if save_csv:
                df.to_csv(f"{name}_SEC.csv", index=False)
            results[name] = df
            print(f"{name}: {len(df)} observations")
            missing = df[["Shares", "STD", "LTD"]].isna().sum()
            if missing.any():
                print(f"  Missing values: {missing[missing > 0].to_dict()}")

        except (KeyError, requests.RequestException, ValueError) as e:
            print(f"Error {company}: {e}")

        time.sleep(0.2)

    return results

In [14]:
email = "sapunkovstepan@gmail.com"
credit = download_credit_data(["BA"], email)

BA: 70 observations


In [15]:
from edgar import Company, set_identity

tickers = ["BA", "AAPL", "TSLA", "F"]
forms = ["10-K", "10-Q"]

set_identity(f"Research {email}")
company_facts = download_credit_data(tickers, email, save_csv=False)
rows = []

for ticker in tickers:
    company = Company(ticker)
    for form in forms:
        filing = company.get_filings(form=form, amendments=False).latest()
        period = str(filing.period_of_report)
        sheet = filing.xbrl().statements.balance_sheet().to_dataframe(standard=False)
        debt = sheet.loc[
            ~sheet["dimension"] & ~sheet["abstract"]
            & sheet["balance"].eq("credit")
            & sheet["label"].str.contains(r"\bdebt\b|borrowings|commercial paper", case=False, na=False)
        ].dropna(subset=[period]).drop_duplicates(["concept", period])
        debt = debt.loc[~debt["parent_concept"].isin(debt["concept"])]

        # Срок долга определяется его разделом в балансе.
        current = debt["parent_concept"].eq("us-gaap_LiabilitiesCurrent")
        noncurrent = (
            debt["concept"].str.contains("Noncurrent|CapitalLeaseObligations$", case=False)
            | debt["parent_concept"].eq("us-gaap_LiabilitiesNoncurrent")
            | debt["label"].str.contains("long.term|net of current|less current", case=False)
        ) & ~current & ~debt["concept"].str.contains("IncludingCurrentMaturities", case=False)

        filed = pd.Timestamp(filing.filing_date)
        baseline = company_facts[ticker].set_index("Date").reindex([filed]).iloc[0]
        rows.append({
            "Ticker": ticker, "Form": filing.form, "Accession": filing.accession_no,
            "Date": filed, "Period": period,
            "STD Company Facts": baseline["STD"],
            "STD EdgarTools": debt.loc[current, period].sum(min_count=1),
            "LTD Company Facts": baseline["LTD"],
            "LTD EdgarTools": debt.loc[noncurrent, period].sum(min_count=1),
        })

debt_check = pd.DataFrame(rows).convert_dtypes().sort_values(["Ticker", "Date"], ignore_index=True)
display(debt_check)

BA: 70 observations
AAPL: 70 observations
TSLA: 62 observations
  Missing values: {'STD': 7, 'LTD': 38}
F: 70 observations
  Missing values: {'Shares': 62, 'STD': 70, 'LTD': 68}


,Ticker,Form,Accession,Date,Period,STD Company Facts,STD EdgarTools,LTD Company Facts,LTD EdgarTools
0,AAPL,10-Q,0000320193-26-000020,2026-07-31,2026-06-27,13004000000,13004000000,71340000000,71340000000
1,AAPL,10-K,0000320193-26-000026,2026-11-03,2026-09-26,11903000000,11903000000,68208000000,68208000000
2,BA,10-K,0001628280-26-004357,2026-01-30,2025-12-31,8461000000,8461000000,45637000000,45637000000
3,BA,10-Q,0001628280-26-067508,2026-10-27,2026-09-30,4569000000,4569000000,41351000000,41351000000
4,F,10-K,0000037996-26-000015,2026-02-11,2025-12-31,<NA>,57302000000,<NA>,106034000000
5,F,10-Q,0000037996-26-000191,2026-10-29,2026-09-30,<NA>,53313000000,<NA>,109769000000
6,TSLA,10-K,0001628280-26-003952,2026-01-29,2025-12-31,1569000000,1640000000,<NA>,6736000000
7,TSLA,10-Q,0001628280-26-066775,2026-10-22,2026-09-30,1311000000,1387000000,<NA>,8136000000


,Ticker,Form,Accession,Date,Period,STD Company Facts,STD EdgarTools,LTD Company Facts,LTD EdgarTools
0,AAPL,10-Q,0000320193-26-000020,2026-07-31,2026-06-27,13004000000,13004000000,71340000000,71340000000
1,AAPL,10-K,0000320193-26-000026,2026-11-03,2026-09-26,11903000000,11903000000,68208000000,68208000000
2,BA,10-K,0001628280-26-004357,2026-01-30,2025-12-31,8461000000,8461000000,45637000000,45637000000
3,BA,10-Q,0001628280-26-067508,2026-10-27,2026-09-30,4569000000,4569000000,41351000000,41351000000
4,F,10-K,0000037996-26-000015,2026-02-11,2025-12-31,<NA>,57302000000,<NA>,106034000000
5,F,10-Q,0000037996-26-000191,2026-10-29,2026-09-30,<NA>,53313000000,<NA>,109769000000
6,TSLA,10-K,0001628280-26-003952,2026-01-29,2025-12-31,1569000000,1640000000,<NA>,6736000000
7,TSLA,10-Q,0001628280-26-066775,2026-10-22,2026-09-30,1311000000,1387000000,<NA>,8136000000
